In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from delta import *

builder = SparkSession.builder \
    .appName("Abhi-M1-Final-Victory") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

try:
    spark.range(10).withColumn("status", f.lit("Permissions Fixed")).write.format("delta").mode("overwrite").save("s3a://lakehouse/victory_table")
    print("🚀 FINALLY! Data is in MinIO.")
    spark.read.format("delta").load("s3a://lakehouse/victory_table").show()
except Exception as e:
    print(f"❌ Error: {e}")

🚀 FINALLY! Data is in MinIO.
+---+-----------------+
| id|           status|
+---+-----------------+
|  5|Permissions Fixed|
|  6|Permissions Fixed|
|  7|Permissions Fixed|
|  8|Permissions Fixed|
|  9|Permissions Fixed|
|  0|Permissions Fixed|
|  1|Permissions Fixed|
|  2|Permissions Fixed|
|  3|Permissions Fixed|
|  4|Permissions Fixed|
+---+-----------------+



In [2]:
# Query the table directly without registering it
df_sql = spark.sql("SELECT * FROM delta.`s3a://lakehouse/victory_table` LIMIT 5")
df_sql.show()

+---+-----------------+
| id|           status|
+---+-----------------+
|  5|Permissions Fixed|
|  6|Permissions Fixed|
|  7|Permissions Fixed|
|  8|Permissions Fixed|
|  9|Permissions Fixed|
+---+-----------------+



2. The Registered View Approach (Recommended)

If you want to write more complex SQL (joins, aggregations), it’s much cleaner to create a Temporary View. This doesn't move the data; it just gives Spark a "nickname" for that S3 location.


In [3]:
# 1. Load the data from MinIO
victory_df = spark.read.format("delta").load("s3a://lakehouse/victory_table")

# 2. Register it as a SQL temporary view
victory_df.createOrReplaceTempView("victory_data")

# 3. Now run standard SQL
query_result = spark.sql("""
    SELECT 
        status, 
        count(*) as row_count,
        MAX(id) as max_id
    FROM victory_data
    GROUP BY status
""")

query_result.show()

+-----------------+---------+------+
|           status|row_count|max_id|
+-----------------+---------+------+
|Permissions Fixed|       10|     9|
+-----------------+---------+------+



In [8]:
# Create 10 rows of test data
df = spark.range(10).withColumn("cluster_node", f.lit("M1-Worker-Active"))

try:
    print("Attempting to write Delta table to MinIO...")
    df.write.format("delta") \
      .mode("overwrite") \
      .save("s3a://lakehouse/verified_production_table")
    
    print("🚀 BOOM! Write Successful. Checking data...")
    
    # Read it back to confirm
    spark.read.format("delta").load("s3a://lakehouse/verified_production_table").show()
    
except Exception as e:
    print(f"❌ Still failing. Error Detail: {e}")

Attempting to write Delta table to MinIO...
🚀 BOOM! Write Successful. Checking data...
+---+----------------+
| id|    cluster_node|
+---+----------------+
|  7|M1-Worker-Active|
|  8|M1-Worker-Active|
|  9|M1-Worker-Active|
|  2|M1-Worker-Active|
|  3|M1-Worker-Active|
|  4|M1-Worker-Active|
|  5|M1-Worker-Active|
|  6|M1-Worker-Active|
|  0|M1-Worker-Active|
|  1|M1-Worker-Active|
+---+----------------+



**3. How to verify the "Delta" Magic**

Since it's a Delta table, you can also look at the History and Metadata using SQL. This is the "Lakehouse" feature that standard CSVs or Parquets don't have.

**Troubleshooting Tip: "Table Not Found"**

If you get a "Table not found" error while using spark.sql, it's usually because:

Backticks: You forgot the backticks ` around the S3 path.

Catalog: You are trying to use a permanent table name (like mydb.table) without having a Hive Metastore set up. For your current Docker setup, Temporary Views (createOrReplaceTempView) are the way to go

In [9]:
!ls -l /usr/local/spark/jars/hadoop-aws-3.3.4.jar

-rw-r--r-- 1 root root 962685 Jul 29  2022 /usr/local/spark/jars/hadoop-aws-3.3.4.jar


In [ ]:
print(spark._jvm.java.lang.System.getProperty("java.class.path"))

In [10]:
!cat /usr/local/spark/conf/spark-defaults.conf

spark.hadoop.fs.s3a.impl           org.apache.hadoop.fs.s3a.S3AFileSystem
spark.hadoop.fs.s3a.endpoint       http://minio-storage:9000
spark.hadoop.fs.s3a.access.key     admin
spark.hadoop.fs.s3a.secret.key     password
spark.hadoop.fs.s3a.path.style.access true
spark.hadoop.fs.s3a.connection.ssl.enabled false
spark.driver.extraClassPath        /usr/local/spark/jars/hadoop-aws-3.3.4.jar:/usr/local/spark/jars/aws-java-sdk-bundle-1.12.262.jar
spark.executor.extraClassPath      /usr/local/spark/jars/hadoop-aws-3.3.4.jar:/usr/local/spark/jars/aws-java-sdk-bundle-1.12.262.jar

